# Register the NeMo TFM Pipeline

Run this notebook **once** from the workbench to compile the pipeline definition
and upload it to the OpenShift AI pipeline server.

After running, you will be able to:
- See the pipeline in the **Pipelines** tab of the RHOAI dashboard
- Create and trigger runs directly from the dashboard UI
- Track each notebook step's logs, inputs, and outputs

---

**Pre-requisite:** the project notebooks must be present at `/opt/app-root/src/`  
If this is a fresh workbench, clone the repo first:
```bash
git clone https://github.com/robbybrodie/transaction-foundation-model-openshiftai.git /opt/app-root/src/nemo-tfm
```

## 1 · Install KFP SDK

The NeMo image's venv is read-only, so we install `kfp` to a folder on the
shared PVC (`/opt/app-root/src/lib`).  This persists across pod restarts and
only needs to run once per workbench — skip this cell if `lib/` already exists.

In [ ]:
import subprocess, sys, os

LIB_DIR = "/opt/app-root/src/lib"

if not os.path.isdir(os.path.join(LIB_DIR, "kfp")):
    print("Installing kfp to PVC lib dir (one-time, ~60s)...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--target", LIB_DIR,
        "--quiet",
        "kfp>=2.7,<3",
        "kfp-kubernetes>=1.3,<2",
    ])
    print("Done.")
else:
    print("kfp already installed in lib dir — skipping.")

# Add the PVC lib dir to the front of the path so it takes precedence
if LIB_DIR not in sys.path:
    sys.path.insert(0, LIB_DIR)

import kfp
print(f"kfp {kfp.__version__} ready")

## 2 · Locate the pipeline YAML

The compiled YAML is committed to the repo — no recompilation needed.
A safety check ensures caching is explicitly disabled before upload.

In [ ]:
import os

PIPELINE_NAME = "nemo-transaction-foundation-model"

# Locate repo root by finding the pipeline YAML
for candidate in [
    "/opt/app-root/src/nemo-tfm",
    "/opt/app-root/src",
    "/opt/app-root/src/transaction-foundation-model-openshiftai",
]:
    yaml_path = os.path.join(candidate, "pipeline", "nemo_tfm_pipeline.yaml")
    if os.path.isfile(yaml_path):
        REPO_ROOT = candidate
        PIPELINE_YAML = yaml_path
        break
else:
    raise FileNotFoundError(
        "Cannot find pipeline/nemo_tfm_pipeline.yaml — "
        "clone the repo to /opt/app-root/src/nemo-tfm first:\n"
        "  git clone https://github.com/robbybrodie/transaction-foundation-model-openshiftai.git "
        "/opt/app-root/src/nemo-tfm"
    )

print(f"Repo root:     {REPO_ROOT}")
print(f"Pipeline YAML: {PIPELINE_YAML}")

# Safety net: ensure enableCache: false is in every cachingOptions block.
# The pre-committed YAML already has this, but guard against stale files.
with open(PIPELINE_YAML) as f:
    content = f.read()

if "enableCache: false" not in content:
    content = content.replace(
        "cachingOptions: {}",
        "cachingOptions:\n          enableCache: false",
    )
    with open(PIPELINE_YAML, "w") as f:
        f.write(content)
    print("Patched YAML: set enableCache=false on all tasks")
else:
    print("Caching already disabled in YAML ✓")

## 3 · Connect to the pipeline server

In [ ]:
import kfp

# In-cluster endpoint — port 8888 is the direct KFP API (HTTPS + SA token).
# Port 8443 is the OAuth proxy which rejects service account tokens; use 8888.
KFP_ENDPOINT = "https://ds-pipeline-pipelines-definition.nemo-tfm.svc.cluster.local:8888"
SA_TOKEN_PATH = "/var/run/secrets/kubernetes.io/serviceaccount/token"

with open(SA_TOKEN_PATH) as f:
    token = f.read().strip()

client = kfp.Client(
    host=KFP_ENDPOINT,
    existing_token=token,
    verify_ssl=False,   # cluster uses internal self-signed cert
)

# Smoke-test — list existing pipelines
existing = client.list_pipelines()
print(f"Connected. Pipelines already registered: {existing.total_size}")

## 4 · Upload (or update) the pipeline

In [ ]:
from datetime import datetime, timezone

# Version name — timestamp so the newest upload is always auto-selected in the UI
version_name = f"v{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M')}"

existing_ids = {
    p.display_name: p.pipeline_id
    for p in (existing.pipelines or [])
}

if PIPELINE_NAME in existing_ids:
    pipeline_id = existing_ids[PIPELINE_NAME]
    result = client.upload_pipeline_version(
        pipeline_package_path=PIPELINE_YAML,
        pipeline_version_name=version_name,
        pipeline_id=pipeline_id,
    )
    print(f"Updated pipeline '{PIPELINE_NAME}' — new version: {version_name}")
    print(f"Version id: {result.pipeline_version_id}")
else:
    result = client.upload_pipeline(
        pipeline_package_path=PIPELINE_YAML,
        pipeline_name=PIPELINE_NAME,
    )
    print(f"Registered new pipeline '{PIPELINE_NAME}'")
    print(f"Pipeline id: {result.pipeline_id}")

print(f"\n✓ Done. In the RHOAI dashboard:")
print(f"  Pipelines tab → {PIPELINE_NAME} → Create run")
print(f"  The version '{version_name}' will be pre-selected (most recent).")

---
## What happens when you run the pipeline?

| Step | Notebook | What it does |
|------|----------|--------------|
| 1 | `01_dataset_baseline.ipynb` | Load TabFormer dataset, temporal splits, XGBoost baseline |
| 2 | `02_seq_preproc_tokenization.ipynb` | GPU-accelerated tokeniser pipeline (cuDF) |
| 3 | `03_foundation_model_training.ipynb` | Pre-train NeMo decoder (30-step demo → full run) |
| 4 | `04_inference_embedding_extraction.ipynb` | Extract 512-d embeddings, UMAP visualisation |
| 5 | `05_xgboost_fraud_detection.ipynb` | Compare XGBoost with raw features vs. embeddings |

Each step's executed notebook is saved to `pipeline-outputs/` on the shared PVC,  
so you can inspect every cell's output after the run — even if you didn't watch it live.